In [1]:
!nvidia-smi

Wed May 27 11:13:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/VLM-vs-OCR-Table-Extraction-Benchmark-on-DocVQA-TableBank")
print("Working dir:", os.getcwd())

Mounted at /content/drive
Working dir: /content/drive/MyDrive/VLM-vs-OCR-Table-Extraction-Benchmark-on-DocVQA-TableBank


In [3]:
# ─── CELL 2: install ────────────────────────────────────────────────────
!pip install transformers accelerate bitsandbytes datasets pillow pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("lmms-lab/DocVQA", "DocVQA", split="validation[:300]")
df_meta = pd.read_csv("results/metadata.csv")

print(f"Dataset: {len(ds)} samples")
print(df_meta.head(3))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.69k [00:00<?, ?B/s]

DocVQA/validation-00000-of-00006.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

DocVQA/validation-00001-of-00006.parquet:   0%|          | 0.00/160M [00:00<?, ?B/s]

DocVQA/validation-00002-of-00006.parquet:   0%|          | 0.00/184M [00:00<?, ?B/s]

...

DocVQA/test-00004-of-00006.parquet:   0%|          | 0.00/211M [00:00<?, ?B/s]

DocVQA/test-00005-of-00006.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/5349 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5188 [00:00<?, ? examples/s]

Dataset: 300 samples
   id  question_id                                           question  \
0   0        49153  What is the ‘actual’ value per 1000, during th...   
1   1        24580                        What is name of university?   
2   2        57349                   What is the name of the company?   

               ground_truth   question_type complexity  
0                      0.28  figure/diagram     simple  
1  university of california          others     medium  
2               itc limited          layout     simple  


In [8]:
# ─── CELL 4: load Qwen2-VL model ────────────────────────────────────────
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    min_pixels=256*28*28,
    max_pixels=512*28*28
)

print("Model loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded.
GPU memory used: 4.42 GB


In [9]:
# ─── CELL 5: run VLM inference on all 300 samples ───────────────────────
import time, torch
import pandas as pd

results = []

for i in range(len(ds)):
    sample = ds[i]
    img    = sample['image']
    gt     = df_meta.loc[i, 'ground_truth']
    question = df_meta.loc[i, 'question']

    # build prompt — pass the actual question for fair comparison
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": f"Answer this question using only the document image. Be brief and precise.\nQuestion: {question}\nAnswer:"}
        ]
    }]

    # tokenise
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text_input], images=[img],
        return_tensors="pt"
    ).to("cuda")

    # inference
    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,    # answers are short — no need for more
            do_sample=False       # greedy decode — faster + deterministic
        )
    elapsed = time.perf_counter() - t0

    # decode only the new tokens (not the prompt)
    input_len  = inputs['input_ids'].shape[1]
    new_tokens = output_ids[0][input_len:]
    vlm_answer = processor.decode(new_tokens, skip_special_tokens=True).strip()

    # check correctness
    vlm_correct = str(gt).lower().strip() in vlm_answer.lower()

    results.append({
        'id':            i,
        'ground_truth':  gt,
        'question':      question,
        'question_type': df_meta.loc[i, 'question_type'],
        'complexity':    df_meta.loc[i, 'complexity'],
        'vlm_answer':    vlm_answer,
        'vlm_correct':   vlm_correct,
        'vlm_time':      round(elapsed, 3),
    })

    # checkpoint every 50
    if i % 50 == 0:
        pd.DataFrame(results).to_csv("results/vlm_checkpoint.csv", index=False)
        acc_so_far = sum(r['vlm_correct'] for r in results) / len(results)
        print(f"Done {i}/300 — acc so far: {acc_so_far:.2f} | last answer: '{vlm_answer}' | gt: '{gt}'")

print("\nAll done!")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Done 0/300 — acc so far: 0.00 | last answer: '0.24' | gt: '0.28'
Done 50/300 — acc so far: 0.75 | last answer: '$200' | gt: '$200'
Done 100/300 — acc so far: 0.77 | last answer: '20 years' | gt: '20 years'
Done 150/300 — acc so far: 0.81 | last answer: '25.9' | gt: '25.9'
Done 200/300 — acc so far: 0.83 | last answer: '$ 3,038,444' | gt: '$ 3,038,444'
Done 250/300 — acc so far: 0.79 | last answer: '11/8/2001' | gt: '11/8/2001'

All done!


In [10]:
# ─── CELL 6: save results ───────────────────────────────────────────────
df_vlm = pd.DataFrame(results)
df_vlm.to_csv("results/vlm_results.csv", index=False)

print("Overall VLM accuracy:", round(df_vlm['vlm_correct'].mean(), 3))
print("\nBy complexity:")
print(df_vlm.groupby('complexity')['vlm_correct'].mean().round(3))
print("\nAvg time per image:", round(df_vlm['vlm_time'].mean(), 2), "s")

Overall VLM accuracy: 0.77

By complexity:
complexity
complex    0.143
medium     0.750
simple     0.801
Name: vlm_correct, dtype: float64

Avg time per image: 0.74 s


In [11]:
# ─── CELL 7: merge with OCR results ─────────────────────────────────────
df_ocr = pd.read_csv("results/ocr_results.csv")

df_compare = df_ocr[['id','ground_truth','complexity',
                      'question_type','raw_correct',
                      'proc_correct','raw_time','proc_time']].merge(
    df_vlm[['id','vlm_answer','vlm_correct','vlm_time']],
    on='id'
)

df_compare.to_csv("results/comparison_table.csv", index=False)
print("Merged comparison table saved.")
print(df_compare[['raw_correct','proc_correct','vlm_correct']].mean().round(3))

Merged comparison table saved.
raw_correct     0.647
proc_correct    0.667
vlm_correct     0.770
dtype: float64


In [ ]:


!git add .
!git commit -m "Add notebook 03 — Qwen2-VL achieves 77% vs OCR 65-67%"
!git push